(line-clipping-section)=
# Line clipping

Since the objects that form our virtual environment are constructed using polygons which are formed using lines, the problem of clipping the screen space can be reduced to the clipping of individual lines. Consider the diagram in {numref}`clipping-lines-figure` which shows the different line clipping scenarios we need to consider.

```{figure} /images/clipping_lines.svg
:width: 400px
:name: clipping-lines-figure

Various line clipping scenarios.
```

Visual inspection of {numref}`clipping-lines-figure` shows we need to do the following

- $\mathbf{a} \to \mathbf{b}$: both endpoints are inside of the visible region so we can draw the line $\mathbf{a} \to \mathbf{b}$;
- $\mathbf{c} \to \mathbf{d}$: the endpoint $\mathbf{c}$ is outside and $\mathbf{d}$ is inside of the visible region so we clip $\mathbf{c}$ to the top edge at $\mathbf{c}'$ and draw the line $\mathbf{c}' \to \mathbf{d}$;
- $\mathbf{e} \to \mathbf{f}$: both endpoints are outside of the visible region and the line $\mathbf{e} \to \mathbf{f}$ does not enter the visible region and is ignored;
- $\mathbf{g} \to \mathbf{h}$: both endpoints are outside of the visible region and the line $\mathbf{g} \to \mathbf{h}$ does enter the visible region so we clip endpoint $\mathbf{g}$ to the bottom edge at $\mathbf{g}'$ and also clip endpoint $\mathbf{h}$ to the right edge at $\mathbf{h}'$ and draw the line $\mathbf{g}' \to \mathbf{h}'$.

## The Cyrus-Beck algorithm

The Cyrus-Beck algorithm {cite:p}`cyrus:1978` is an algorithm for clipping a single line to a single edge. Consider the diagram in {numref}`clip-to-edge-figure` which shows the line joining two points $\mathbf{a} \to \mathbf{b}$ that passes through the plane defined by the [normal vector](normal-vector-section) $\mathbf{n}$ and point $\mathbf{p}$.


```{figure} /images/clip_to_edge.svg
:width: 300px
:name: clip-to-edge-figure

The line joining points $\mathbf{a} \to \mathbf{b}$ is clipped to the left edge at point $\mathbf{c}$.
```

Clipping of the line joining points $\mathbf{a}$ and $\mathbf{b}$ to the plane requires the calculation of the point where the line intersects with the plane. To calculate the intersection point $\mathbf{c}$ we use the vector formula for calculating the shortest distance between a point and a plane, equation {eq}`point-plane-distance-equation`

```{math}
:label: cyrus-beck-d-equation

d = (\mathbf{q} - \mathbf{p}) \cdot \frac{\mathbf{n}}{|\mathbf{n}|}.
```

The vector equation of the line $\mathbf{a} \to \mathbf{b}$ is

```{math}
:label: cyrus-beck-q-equation

\mathbf{q} = \mathbf{a} + t(\mathbf{b} - \mathbf{a}),
```

and substituting into equation {eq}`cyrus-beck-d-equation`

\begin{align*}
    d &= (\mathbf{a} + t(\mathbf{b} - \mathbf{a}) - \mathbf{p}) \cdot \frac{\mathbf{n}}{|\mathbf{n}|}.
\end{align*}

The intersection point $\mathbf{c}$ is where the distance to the plane is zero so setting $d=0$ and rearranging to make $t$ the subject gives

```{math}
:label: cyrus-beck-t-equation-1

\begin{align}
     0 &= (\mathbf{a} + t(\mathbf{b} - \mathbf{a}) - \mathbf{p}) \cdot \frac{\mathbf{n}}{|\mathbf{n}|}\\
     &= (\mathbf{a} - \mathbf{p}) \cdot \mathbf{n} + t(\mathbf{b} - \mathbf{a}) \cdot \mathbf{n} \\
     t(\mathbf{a} - \mathbf{b}) \cdot \mathbf{n} &= (\mathbf{a} - \mathbf{p}) \cdot \mathbf{n} \\
     t &= \frac{(\mathbf{a} - \mathbf{p}) \cdot \mathbf{n}}{(\mathbf{a} - \mathbf{b}) \cdot \mathbf{n}}
\end{align}
```

This expression for $t$ can be substituted into equation {eq}`cyrus-beck-q-equation` to give $\mathbf{c}$. 

On visual inspection it is easy for us to see that in {numref}`clip-to-edge-figure` point $\mathbf{a}$ is outside of the visible region and needs to be clipped to the left edge at $\mathbf{c}$ and the line $\mathbf{a}$ to $\mathbf{c}$ is replaced by $\mathbf{c}$ to $\mathbf{b}$. To get a computer to do this we can use the distance between the endpoints and the edges of the visible region. If the normal vectors of the edges of the visible region are pointing into the interior, then any point where the distance to the edge calculated using equation {eq}`point-plane-distance-equation` is negative is outside and needs to be clipped. So before calculating $t$ we calculate the distances of $\mathbf{a}$ and $\mathbf{b}$ to the edge

\begin{align*}
    d_\mathbf{a} &= (\mathbf{a} - \mathbf{p}) \cdot \mathbf{n}, \\
    d_\mathbf{b} &= (\mathbf{b} - \mathbf{p}) \cdot \mathbf{n}
\end{align*}

There are four possible situations to consider

- If $d_\mathbf{a} < 0$ and $d_\mathbf{b} > 0$ then $\mathbf{a}$ is outside and $\mathbf{b}$ is inside and we clip $\mathbf{a}$ to $\mathbf{c}$;
- Else if $d_\mathbf{a} > 0$ and $d_\mathbf{b} < 0$ then $\mathbf{a}$ is inside and $\mathbf{b}$ is outside and we clip $\mathbf{b}$ to $\mathbf{c}$;
- Else if $d_\mathbf{a} > 0$ and $d_\mathbf{b} > 0$ then both $\mathbf{a} \to \mathbf{b}$ are inside so we don't need to clip the line joining $\mathbf{a} \to \mathbf{b}$;
- Else if $d_\mathbf{a} < 0$ and $d_\mathbf{b} < 0$ then both $\mathbf{a} \to \mathbf{b}$ are outside so we ignore the line joining $\mathbf{a} \to \mathbf{b}$.

Since we have calculated $d_\mathbf{a}$ and $d_\mathbf{b}$ to determine whether clipping is required, it makes sense to use these values to calculate $t$. Rearranging equation {eq}`cyrus-beck-t-equation-1` gives

```{math}
:label: cyrus-beck-t-equation-2

\begin{align}
    t &= \frac{d_\mathbf{a}}{d_\mathbf{a} - d_\mathbf{b}}.
\end{align}
```

```{prf:algorithm} Cyrus-Beck algorithm
:label: cyrus-beck-algorithm

**Inputs** An edge of the visible region defined by its inwardly pointing normal vector $\mathbf{n}$ and a point on the edge $\mathbf{p}$ and a line with endpoint $\mathbf{a} \to \mathbf{b}$.

**Outputs** Two endpoints of a line $\mathbf{a} \to \mathbf{b}$.

- $d_\mathbf{a} \gets (\mathbf{a} - \mathbf{p}) \cdot \mathbf{n}$ and $d_\mathbf{b} \gets (\mathbf{b} - \mathbf{p}) \cdot \mathbf{n}$
- $t \gets \dfrac{d_\mathbf{a}}{d_\mathbf{a} - d_\mathbf{b}}$
- If $d_\mathbf{a} < 0$ and $d_\mathbf{b} > 0$
    - $\mathbf{a} \gets \mathbf{a} + t (\mathbf{b} - \mathbf{a})$
- Else if $d_\mathbf{a} > 0$ and $d_\mathbf{b} < 0$
    - $\mathbf{b} \gets \mathbf{a} + t (\mathbf{b} - \mathbf{a})$
- Else if $d_\mathbf{a} < 0$ and $d_\mathbf{b} < 0$
    - $\mathbf{b} \gets \mathbf{a}$ &emsp; (creates a line of zero length)
- Return $\mathbf{a}$ and $\mathbf{b}$
```

`````{prf:example}
:class: seealso
:label: cyrus-beck-example

Use the Cyrus-Beck algorithm to clip the lines $\mathbf{a} \to \mathbf{b}$, $\mathbf{c} \to \mathbf{d}$, $\mathbf{e} \to \mathbf{f}$ and $\mathbf{g} \to \mathbf{h}$ in the diagram below.

```{figure} /images/cyrus_beck_example.svg
```

````{dropdown} Solution
The normal vectors for the four edges are: 
\begin{align*}
    \mathbf{n}_{left} &= \begin{pmatrix} 1 \\ 0 \end{pmatrix}, &
    \mathbf{n}_{bottom} &= \begin{pmatrix} 0 \\ 1 \end{pmatrix}, \\
    \mathbf{n}_{right} &= \begin{pmatrix} -1 \\ 0 \end{pmatrix}, &
    \mathbf{n}_{top} &= \begin{pmatrix} 0 \\ -1 \end{pmatrix}.
\end{align*}

Line $\mathbf{a} \to \mathbf{b}$: considering each edge in turn

- bottom: $d_\mathbf{a} = 1 > 0$ and $d_\mathbf{b} = 3 > 0$ so do not clip $\mathbf{a} \to \mathbf{b}$
- right: $d_\mathbf{a} = 5 > 0$ and $d_\mathbf{d} = 4 > 0$ so do not clip $\mathbf{a} \to \mathbf{b}$
- top: $d_\mathbf{a} = 3 > 0$ and $d_\mathbf{d} = 1 > 0$ so do not clip $\mathbf{a} \to \mathbf{b}$
- left: $d_\mathbf{a} = 1 > 0$ and $d_\mathbf{d} = 2 > 0$ so do not clip $\mathbf{a} \to \mathbf{b}$

Line $\mathbf{c} \to \mathbf{d}$:

- bottom: $d_\mathbf{c} = 5 > 0$ and $d_\mathbf{d} = 3 > 0$ so do not clip $\mathbf{c} \to \mathbf{d}$
- right: $d_\mathbf{c} = 3 > 0$ and $d_\mathbf{d} = 2 > 0$ so do not clip $\mathbf{c} \to \mathbf{d}$
- top: $d_\mathbf{c} = -1 < 0$ and $d_\mathbf{d} = 1 > 0$ so clip $\mathbf{c}$ to the top edge

\begin{align*}
    t &= \frac{d_\mathbf{c}}{d_\mathbf{c} - d_\mathbf{d}} = \frac{-1}{-1 - 1} = \frac{1}{2},  \\
    \mathbf{c}' &= \mathbf{c} + t(\mathbf{d} - \mathbf{c}), 
    = \begin{pmatrix} 6 \\ 7 \end{pmatrix} + \frac{1}{2} \left(
        \begin{pmatrix} 7 \\ 5 \end{pmatrix} -
        \begin{pmatrix} 6 \\ 7 \end{pmatrix}
    \right)
    = \begin{pmatrix} 13/2 \\ 6 \end{pmatrix}.
\end{align*}

- left: $d_{\mathbf{c}'} = 0$ and $d_\mathbf{d} > 0$ so do not clip $\mathbf{c}' \to \mathbf{d}$

Line $\mathbf{e} \to \mathbf{f}$:

- bottom: $d_\mathbf{e} = 3 > 0$ and $d_\mathbf{f} = 5 > 0$ so do not clip $\mathbf{e} \to \mathbf{f}$
- right: $d_\mathbf{e} = 8 > 0$ and $d_\mathbf{f} = 7 > 0$ so do not clip $\mathbf{e} \to \mathbf{f}$
- top: $d_\mathbf{e} = 1 > 0$ and $d_\mathbf{f} = -1 < 0$ so clip $\mathbf{f}$ to the top edge

\begin{align*}
    t &= \frac{d_{\mathbf{e}}}{d_{\mathbf{e}} - d_{\mathbf{f}}} = \frac{1}{1 -(-1)} = \frac{1}{2}, \\
    \mathbf{f}' &= \mathbf{e} + t(\mathbf{f} - \mathbf{e})
    = \begin{pmatrix} 1 \\ 5 \end{pmatrix} + \frac{1}{2} \left(
        \begin{pmatrix} 2 \\ 7 \end{pmatrix} -
        \begin{pmatrix} 1 \\ 5 \end{pmatrix}
    \right)
    = \begin{pmatrix} 3 / 2 \\ 6 \end{pmatrix}
\end{align*}

- left: $d_{\mathbf{e}} = -2 < 0$ and $d_{\mathbf{f}'} = -3/2 < 0$ so do not draw line $\mathbf{e} \to \mathbf{f}'$.

````

`````